# Driver Performance Analysis

**Phase 3 — Analytics | F1 Data Platform**

Analyses driver pace, consistency, racecraft, and standings using Gold mart tables only.

**Key questions:**
1. How do drivers compare on outright pace?
2. Who is most consistent lap-to-lap?
3. Who gains or loses the most positions in races?
4. How do teammates compare head-to-head?
5. Where do drivers stand in the championship?

**Data:** 2026 season — rounds loaded are queried dynamically at runtime.

## 0. Setup

In [1]:
import sys
from pathlib import Path

# Repo root is two levels up from notebooks/analytics/
repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root))

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dotenv import load_dotenv

load_dotenv(repo_root / ".env")

# ── Parameters ────────────────────────────────────────────────────────────────
SEASON = 2026
DB_PATH = repo_root / "data" / "f1_local.duckdb"
PLOTLY_TEMPLATE = "plotly_dark"

# Keyed on team_id (mart_lap_times) and constructor (mart_race_results) —
# both use the same lowercase slug. Add/update for 2026 liveries as needed.
TEAM_COLORS = {
    "red_bull":         "#3671C6",
    "ferrari":          "#E8002D",
    "mercedes":         "#27F4D2",
    "mclaren":          "#FF8000",
    "aston_martin":     "#229971",
    "alpine":           "#FF87BC",
    "haas_f1_team":     "#B6BABD",
    "rb":               "#6692FF",
    "williams":         "#64C4FF",
    "kick_sauber":      "#52E252",
    "cadillac_f1_team": "#C8A84B",
}

print(f"Repo root : {repo_root}")
print(f"DB path   : {DB_PATH}")
print(f"Season    : {SEASON}")

Repo root : /Users/christiangrier/Documents/formula-1-data-project
DB path   : /Users/christiangrier/Documents/formula-1-data-project/data/f1_local.duckdb
Season    : 2026


In [2]:
# ── DuckDB connection — read-only ─────────────────────────────────────────────
con = duckdb.connect(str(DB_PATH), read_only=True)

gold_tables = con.execute(
    "SELECT table_name FROM information_schema.tables "
    "WHERE table_schema = 'gold' ORDER BY table_name"
).df()
print("Gold tables:")
print(gold_tables.to_string(index=False))

Gold tables:
                table_name
mart_constructor_standings
     mart_driver_standings
            mart_lap_times
            mart_pit_stops
         mart_race_results
     mart_telemetry_stints
        mart_tyre_strategy
              mart_weather


In [3]:
# ── Discover loaded rounds ────────────────────────────────────────────────────
rounds_loaded = con.execute(f"""
    SELECT DISTINCT round
    FROM gold.mart_race_results
    WHERE season = {SEASON}
    ORDER BY round
""").df()["round"].tolist()

latest_round = max(rounds_loaded)
print(f"Rounds loaded for {SEASON}: {rounds_loaded}")

Rounds loaded for 2026: [1, 2]


## 1. Load Gold Data

In [4]:
# ── mart_race_results ─────────────────────────────────────────────────────────
# Key cols: abbreviation, constructor, grid_position, finish_position,
#           points, status, round, race_name
race_results = con.execute(f"""
    SELECT *
    FROM gold.mart_race_results
    WHERE season = {SEASON}
    ORDER BY round, finish_position
""").df()

print(f"mart_race_results: {race_results.shape[0]} rows, {race_results['round'].nunique()} rounds")
race_results.head(3)

mart_race_results: 44 rows, 2 rounds


,result_id,season,round,driver_id,driver_number,abbreviation,full_name,team_id,team_name,team_color,...,race_date,is_sprint_weekend,grid_position,finish_position,classified_position,laps,race_time_seconds,points,status,fastest_lap_rank
0,f08a4588c6c8ed01f712ffac933adfc8,2026,1,russell,63,RUS,George Russell,mercedes,Mercedes,00D7B6,...,2026-03-08,False,1,1,1,58,4986.801,25.0,Finished,6
1,30f8bc62eef74e82f5c22204fc990a18,2026,1,antonelli,12,ANT,Kimi Antonelli,mercedes,Mercedes,00D7B6,...,2026-03-08,False,2,2,2,58,2.974,18.0,Finished,3
2,ceeb93105bb8543e9cc5cae92e7712fb,2026,1,leclerc,16,LEC,Charles Leclerc,ferrari,Ferrari,ED1131,...,2026-03-08,False,4,3,3,58,15.519,15.0,Finished,5


In [5]:
# ── mart_lap_times ────────────────────────────────────────────────────────────
# Filters applied here per CLAUDE.md (mart/notebook level, not Silver):
#   is_accurate = TRUE            — telemetry-validated laps only
#   lap_number > 1                — exclude formation lap
#   pit_out_time_seconds IS NULL  — exclude pit outlaps
#                                   (no is_pit_out_lap col in this mart)
lap_times = con.execute(f"""
    SELECT *
    FROM gold.mart_lap_times
    WHERE season = {SEASON}
      AND is_accurate = TRUE
      AND lap_number > 1
      AND pit_out_time_seconds IS NULL
    ORDER BY round, abbreviation, lap_number
""").df()

print(f"mart_lap_times (filtered): {lap_times.shape[0]} rows")
lap_times.head(3)

mart_lap_times (filtered): 1651 rows


,lap_id,season,round,abbreviation,driver_id,full_name,team_id,team_name,team_color,lap_number,...,speed_st,compound,tyre_life,fresh_tyre,is_accurate,is_personal_best,track_status,track_position,pit_out_time_seconds,pit_in_time_seconds
0,3db51696bcce9bcf9a6d518bf036655f,2026,1,ALB,albon,Alexander Albon,williams,Williams,1868DB,2,...,265.0,MEDIUM,2,True,True,True,1,13,NaN,NaN
1,b5c62c27fe932d47229c981d2b3cf458,2026,1,ALB,albon,Alexander Albon,williams,Williams,1868DB,3,...,251.0,MEDIUM,3,True,True,True,1,13,NaN,NaN
2,1002b71ae11928fdb923a0dbed96773b,2026,1,ALB,albon,Alexander Albon,williams,Williams,1868DB,4,...,276.0,MEDIUM,4,True,True,False,1,14,NaN,NaN


In [6]:
# ── mart_driver_standings ─────────────────────────────────────────────────────
standings = con.execute(f"""
    SELECT *
    FROM gold.mart_driver_standings
    WHERE season = {SEASON}
    ORDER BY round, championship_position
""").df()

standings_latest = standings[standings["round"] == latest_round].copy()
print(f"Standings after round {latest_round}: {len(standings_latest)} drivers")
standings_latest[
    ["championship_position", "abbreviation", "driver_name", "constructor", "championship_points", "wins"]
].head(10)

Standings after round 2: 22 drivers


,championship_position,abbreviation,driver_name,constructor,championship_points,wins
0,1,RUS,George Russell,mercedes,51.0,1
1,2,ANT,Andrea Kimi Antonelli,mercedes,47.0,1
2,3,LEC,Charles Leclerc,ferrari,34.0,0
3,4,HAM,Lewis Hamilton,ferrari,33.0,0
4,5,BEA,Oliver Bearman,haas_f1_team,17.0,0
5,6,NOR,Lando Norris,mclaren,15.0,0
6,7,GAS,Pierre Gasly,alpine_f1_team,9.0,0
7,8,VER,Max Verstappen,red_bull,8.0,0
8,9,LAW,Liam Lawson,rb_f1_team,8.0,0
9,10,LIN,Arvid Lindblad,rb_f1_team,4.0,0


## 2. Championship Standings

In [7]:
# ── 2.1 Points after latest round ────────────────────────────────────────────
fig = px.bar(
    standings_latest.sort_values("championship_points", ascending=True),
    x="championship_points",
    y="abbreviation",
    orientation="h",
    color="constructor",
    color_discrete_map=TEAM_COLORS,
    title=f"Driver Championship Standings — {SEASON} (after Round {latest_round})",
    labels={
        "championship_points": "Points",
        "abbreviation": "",
        "constructor": "Constructor",
    },
    template=PLOTLY_TEMPLATE,
    text="championship_points",
)
fig.update_traces(textposition="outside")
fig.update_layout(height=600)
fig.show()

In [8]:
# ── 2.2 Points progression round by round ────────────────────────────────────
if len(rounds_loaded) >= 2:
    fig = px.line(
        standings.sort_values(["abbreviation", "round"]),
        x="round",
        y="championship_points",
        color="abbreviation",
        markers=True,
        title=f"Points Progression — {SEASON}",
        labels={
            "round": "Round",
            "championship_points": "Cumulative Points",
            "abbreviation": "Driver",
        },
        template=PLOTLY_TEMPLATE,
    )
    fig.update_layout(height=500)
    fig.show()
else:
    print("Skipping points progression — need at least 2 rounds.")

## 3. Race Pace

In [9]:
# ── 3.1 Pace summary per driver per round ────────────────────────────────────
# Median is more robust than mean — resistant to SC/VSC distortion.
pace_summary = (
    lap_times
    .groupby(["round", "abbreviation", "team_id"])["lap_time_seconds"]
    .agg(
        median_lap_s="median",
        std_lap_s="std",
        lap_count="count",
    )
    .reset_index()
)

fastest_per_round = pace_summary.groupby("round")["median_lap_s"].transform("min")
pace_summary["delta_to_fastest_s"] = (pace_summary["median_lap_s"] - fastest_per_round).round(3)

pace_summary.sort_values(["round", "delta_to_fastest_s"]).head(10)

,round,abbreviation,team_id,median_lap_s,std_lap_s,lap_count,delta_to_fastest_s
2,1,ANT,mercedes,83.0170,0.862041,49,0.000
16,1,RUS,mercedes,83.0900,0.842625,50,0.073
11,1,LEC,ferrari,83.1670,0.838604,48,0.150
9,1,HAM,ferrari,83.3135,0.818851,48,0.297
13,1,NOR,mclaren,83.5340,1.090390,49,0.517
19,1,VER,red_bull,83.5345,1.104046,48,0.518
3,1,BEA,haas,84.6270,0.806878,49,1.610
4,1,BOR,audi,84.6940,0.935952,49,1.677
12,1,LIN,rb,84.8625,0.893176,48,1.846
10,1,LAW,rb,85.1730,1.308652,49,2.156


In [10]:
# ── 3.2 Pace delta vs fastest — per round ────────────────────────────────────
for rnd in sorted(rounds_loaded):
    race_name = race_results[race_results["round"] == rnd]["race_name"].iloc[0]
    df_rnd = pace_summary[pace_summary["round"] == rnd].sort_values("delta_to_fastest_s")

    fig = px.bar(
        df_rnd,
        x="abbreviation",
        y="delta_to_fastest_s",
        color="team_id",
        color_discrete_map=TEAM_COLORS,
        title=f"Median Race Pace Delta vs Fastest — {race_name} (Round {rnd})",
        labels={
            "abbreviation": "Driver",
            "delta_to_fastest_s": "Gap to fastest (s)",
            "team_id": "Team",
        },
        template=PLOTLY_TEMPLATE,
        text="delta_to_fastest_s",
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(height=450, xaxis_tickangle=-45)
    fig.show()

## 4. Consistency

In [11]:
# ── 4.1 Lap time distribution — box plot per driver ───────────────────────────
# Narrow IQR = consistent. Drivers ordered fastest to slowest median.
for rnd in sorted(rounds_loaded):
    race_name = race_results[race_results["round"] == rnd]["race_name"].iloc[0]
    df_rnd = lap_times[lap_times["round"] == rnd].copy()

    order = (
        df_rnd.groupby("abbreviation")["lap_time_seconds"]
        .median()
        .sort_values()
        .index.tolist()
    )

    fig = px.box(
        df_rnd,
        x="abbreviation",
        y="lap_time_seconds",
        color="team_id",
        color_discrete_map=TEAM_COLORS,
        category_orders={"abbreviation": order},
        title=f"Lap Time Distribution — {race_name} (Round {rnd})",
        labels={
            "abbreviation": "Driver",
            "lap_time_seconds": "Lap time (s)",
            "team_id": "Team",
        },
        template=PLOTLY_TEMPLATE,
        points=False,
    )
    fig.update_layout(height=500, xaxis_tickangle=-45)
    fig.show()

In [12]:
# ── 4.2 Pace vs Consistency scatter ──────────────────────────────────────────
# Bottom-left quadrant = fast AND consistent (ideal)
for rnd in sorted(rounds_loaded):
    race_name = race_results[race_results["round"] == rnd]["race_name"].iloc[0]
    df_rnd = pace_summary[pace_summary["round"] == rnd].copy()

    fig = px.scatter(
        df_rnd,
        x="median_lap_s",
        y="std_lap_s",
        color="team_id",
        color_discrete_map=TEAM_COLORS,
        text="abbreviation",
        title=(
            f"Pace vs Consistency — {race_name} (Round {rnd})<br>"
            "<sup>Bottom-left = fast &amp; consistent | Top-right = slow &amp; erratic</sup>"
        ),
        labels={
            "median_lap_s": "Median lap time (s)  ← faster",
            "std_lap_s": "Std deviation (s)  ← more consistent",
            "team_id": "Team",
        },
        template=PLOTLY_TEMPLATE,
    )
    fig.update_traces(textposition="top center", marker_size=10)
    fig.update_layout(height=500)
    fig.show()

## 5. Racecraft — Positions Gained / Lost

In [13]:
# ── 5.1 Compute positions gained ─────────────────────────────────────────────
# Excludes rows where either position is null (DNF / DNS)
racecraft = race_results.dropna(subset=["grid_position", "finish_position"]).copy()
racecraft["positions_gained"] = racecraft["grid_position"] - racecraft["finish_position"]

racecraft_agg = (
    racecraft
    .groupby(["abbreviation", "constructor"])
    .agg(
        total_positions_gained=("positions_gained", "sum"),
        races=("round", "count"),
        avg_finish=("finish_position", "mean"),
        avg_grid=("grid_position", "mean"),
    )
    .reset_index()
    .sort_values("total_positions_gained", ascending=False)
)

racecraft_agg

,abbreviation,constructor,total_positions_gained,races,avg_finish,avg_grid
19,SAI,williams,14,2,12.0,19.0
3,BEA,haas_f1_team,10,2,6.0,11.0
16,PER,cadillac_f1_team,8,2,15.5,19.5
20,STR,aston_martin,7,2,17.5,21.0
21,VER,red_bull,6,2,11.0,14.0
5,BOT,cadillac_f1_team,6,2,16.0,19.0
7,GAS,alpine_f1_team,5,2,8.0,10.5
6,COL,alpine_f1_team,4,2,12.0,14.0
13,LIN,rb_f1_team,4,2,10.0,12.0
0,ALB,williams,3,2,17.0,18.5


In [14]:
# ── 5.2 Positions gained bar chart ───────────────────────────────────────────
fig = px.bar(
    racecraft_agg.sort_values("total_positions_gained"),
    x="total_positions_gained",
    y="abbreviation",
    orientation="h",
    color="constructor",
    color_discrete_map=TEAM_COLORS,
    title=f"Positions Gained / Lost — {SEASON} (Rounds {rounds_loaded[0]}–{rounds_loaded[-1]})",
    labels={
        "total_positions_gained": "Net positions (+ = forward)",
        "abbreviation": "",
        "constructor": "Constructor",
    },
    template=PLOTLY_TEMPLATE,
    text="total_positions_gained",
)
fig.add_vline(x=0, line_dash="dash", line_color="white", opacity=0.4)
fig.update_traces(textposition="outside")
fig.update_layout(height=600)
fig.show()

In [15]:
# ── 5.3 Grid vs Finish scatter — per round ───────────────────────────────────
# Below diagonal = positions gained. Axes inverted so P1 is top-left.
for rnd in sorted(rounds_loaded):
    race_name = race_results[race_results["round"] == rnd]["race_name"].iloc[0]
    df_rnd = racecraft[racecraft["round"] == rnd].copy()

    fig = px.scatter(
        df_rnd,
        x="grid_position",
        y="finish_position",
        color="constructor",
        color_discrete_map=TEAM_COLORS,
        text="abbreviation",
        title=(
            f"Grid vs Finish — {race_name} (Round {rnd})<br>"
            "<sup>Below diagonal = positions gained | Above = positions lost</sup>"
        ),
        labels={
            "grid_position": "Grid position",
            "finish_position": "Finish position",
            "constructor": "Constructor",
        },
        template=PLOTLY_TEMPLATE,
    )
    max_pos = int(max(df_rnd["grid_position"].max(), df_rnd["finish_position"].max())) + 1
    fig.add_shape(
        type="line", x0=1, y0=1, x1=max_pos, y1=max_pos,
        line=dict(color="white", dash="dash", width=1),
    )
    fig.update_xaxes(autorange="reversed")
    fig.update_yaxes(autorange="reversed")
    fig.update_traces(textposition="top center", marker_size=10)
    fig.update_layout(height=520)
    fig.show()

## 6. Teammate Head-to-Head

In [16]:
# ── 6.1 Finish position H2H ───────────────────────────────────────────────────
h2h_base = race_results[
    race_results["finish_position"].notna()
][["round", "constructor", "abbreviation", "finish_position"]].copy()

h2h = (
    h2h_base
    .merge(h2h_base, on=["round", "constructor"], suffixes=("_a", "_b"))
    .query("abbreviation_a < abbreviation_b")  # canonical pair only
    .assign(
        winner=lambda df: df.apply(
            lambda r: r["abbreviation_a"]
            if r["finish_position_a"] < r["finish_position_b"]
            else r["abbreviation_b"],
            axis=1,
        )
    )
)

h2h_summary = (
    h2h
    .groupby(["constructor", "abbreviation_a", "abbreviation_b", "winner"])
    .size()
    .reset_index(name="rounds_ahead")
    .sort_values(["constructor", "rounds_ahead"], ascending=[True, False])
)

print("Teammate H2H — finish position:")
h2h_summary

Teammate H2H — finish position:


,constructor,abbreviation_a,abbreviation_b,winner,rounds_ahead
0,alpine_f1_team,COL,GAS,GAS,2
1,aston_martin,ALO,STR,ALO,1
2,aston_martin,ALO,STR,STR,1
3,audi,BOR,HUL,BOR,1
4,audi,BOR,HUL,HUL,1
5,cadillac_f1_team,BOT,PER,BOT,1
6,cadillac_f1_team,BOT,PER,PER,1
7,ferrari,HAM,LEC,HAM,1
8,ferrari,HAM,LEC,LEC,1
9,haas_f1_team,BEA,OCO,BEA,2


In [17]:
# ── 6.2 Teammate median pace delta — per round ───────────────────────────────
constructor_map = (
    race_results[["round", "abbreviation", "constructor"]].drop_duplicates()
)

pace_with_team = pace_summary.merge(constructor_map, on=["round", "abbreviation"])

pace_pairs = (
    pace_with_team
    .merge(pace_with_team, on=["round", "constructor"], suffixes=("_a", "_b"))
    .query("abbreviation_a < abbreviation_b")
    .assign(
        pace_delta_s=lambda df: (df["median_lap_s_a"] - df["median_lap_s_b"]).abs().round(3),
        faster=lambda df: df.apply(
            lambda r: r["abbreviation_a"]
            if r["median_lap_s_a"] < r["median_lap_s_b"]
            else r["abbreviation_b"],
            axis=1,
        ),
    )
    [["round", "constructor", "abbreviation_a", "abbreviation_b",
      "median_lap_s_a", "median_lap_s_b", "pace_delta_s", "faster"]]
)

pace_pairs

,round,constructor,abbreviation_a,abbreviation_b,median_lap_s_a,median_lap_s_b,pace_delta_s,faster
1,1,williams,ALB,SAI,85.3340,85.6490,0.315,ALB
3,1,aston_martin,ALO,STR,87.4495,87.4160,0.034,STR
5,1,mercedes,ANT,RUS,83.0170,83.0900,0.073,ANT
7,1,haas_f1_team,BEA,OCO,84.6270,85.2490,0.622,BEA
10,1,cadillac_f1_team,BOT,PER,87.8040,87.5950,0.209,PER
12,1,alpine_f1_team,COL,GAS,85.6330,85.2020,0.431,GAS
16,1,red_bull,HAD,VER,85.3630,83.5345,1.828,VER
18,1,ferrari,HAM,LEC,83.3135,83.1670,0.147,LEC
20,1,rb_f1_team,LAW,LIN,85.1730,84.8625,0.311,LIN
39,2,aston_martin,ALO,STR,100.3560,101.0830,0.727,ALO


In [18]:
# ── 6.3 Pace delta bar chart — per round ─────────────────────────────────────
for rnd in sorted(rounds_loaded):
    race_name = race_results[race_results["round"] == rnd]["race_name"].iloc[0]
    df_rnd = pace_pairs[pace_pairs["round"] == rnd].copy()
    df_rnd["matchup"] = df_rnd["abbreviation_a"] + " vs " + df_rnd["abbreviation_b"]
    df_rnd = df_rnd.sort_values("pace_delta_s", ascending=False)

    fig = px.bar(
        df_rnd,
        x="pace_delta_s",
        y="matchup",
        orientation="h",
        color="faster",
        title=(
            f"Teammate Median Pace Gap — {race_name} (Round {rnd})<br>"
            "<sup>Colour = faster driver</sup>"
        ),
        labels={
            "pace_delta_s": "Pace delta (s)",
            "matchup": "",
            "faster": "Faster driver",
        },
        template=PLOTLY_TEMPLATE,
        text="pace_delta_s",
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(height=500)
    fig.show()

## 7. Race Pace Evolution

In [19]:
# ── 7.1 Lap time by lap number — top 10 finishers only ───────────────────────
# 3-lap rolling average smooths noise while preserving fuel load trend,
# tyre degradation shape, and SC/VSC disturbances.
for rnd in sorted(rounds_loaded):
    race_name = race_results[race_results["round"] == rnd]["race_name"].iloc[0]

    top10 = (
        race_results[
            (race_results["round"] == rnd) &
            (race_results["finish_position"] <= 10)
        ]["abbreviation"].tolist()
    )

    df_rnd = (
        lap_times[
            (lap_times["round"] == rnd) &
            (lap_times["abbreviation"].isin(top10))
        ]
        .sort_values(["abbreviation", "lap_number"])
        .copy()
    )

    df_rnd["lap_time_rolling"] = (
        df_rnd
        .groupby("abbreviation")["lap_time_seconds"]
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )

    fig = px.line(
        df_rnd,
        x="lap_number",
        y="lap_time_rolling",
        color="abbreviation",
        title=f"Race Pace Evolution (3-lap rolling avg, top 10) — {race_name} (Round {rnd})",
        labels={
            "lap_number": "Lap",
            "lap_time_rolling": "Lap time (s)",
            "abbreviation": "Driver",
        },
        template=PLOTLY_TEMPLATE,
    )
    fig.update_layout(height=500)
    fig.show()

## 8. Driver Scorecard

In [20]:
# ── 8.1 Season summary table ──────────────────────────────────────────────────
results_agg = (
    race_results
    .groupby(["abbreviation", "constructor"])
    .agg(
        races=("round", "count"),
        total_points=("points", "sum"),
        wins=("finish_position", lambda x: (x == 1).sum()),
        podiums=("finish_position", lambda x: (x <= 3).sum()),
        avg_finish=("finish_position", "mean"),
        avg_grid=("grid_position", "mean"),
    )
    .reset_index()
)

pace_agg = (
    pace_summary
    .groupby("abbreviation")
    .agg(
        avg_median_pace_s=("median_lap_s", "mean"),
        avg_std_s=("std_lap_s", "mean"),
        avg_delta_to_fastest_s=("delta_to_fastest_s", "mean"),
    )
    .reset_index()
)

scorecard = (
    results_agg
    .merge(pace_agg, on="abbreviation", how="left")
    .merge(
        racecraft_agg[["abbreviation", "total_positions_gained"]],
        on="abbreviation",
        how="left",
    )
    .sort_values("total_points", ascending=False)
)

for col in ["avg_finish", "avg_grid", "avg_median_pace_s", "avg_std_s", "avg_delta_to_fastest_s"]:
    scorecard[col] = scorecard[col].round(2)

print(f"Driver Scorecard — {SEASON} (Rounds {rounds_loaded[0]}–{rounds_loaded[-1]})")
scorecard

Driver Scorecard — 2026 (Rounds 1–2)


,abbreviation,constructor,races,total_points,wins,podiums,avg_finish,avg_grid,avg_median_pace_s,avg_std_s,avg_delta_to_fastest_s,total_positions_gained
2,ANT,mercedes,2,43.0,1,2,1.5,1.5,89.73,0.85,0.02,0
18,RUS,mercedes,2,43.0,1,2,1.5,1.5,89.75,0.94,0.04,0
9,HAM,ferrari,2,27.0,0,1,3.5,5.0,90.09,0.85,0.38,3
12,LEC,ferrari,2,27.0,0,1,3.5,4.0,90.00,0.83,0.29,1
3,BEA,haas_f1_team,2,16.0,0,0,6.0,11.0,91.06,0.91,1.35,10
14,NOR,mclaren,2,10.0,0,0,12.5,6.0,83.53,1.09,0.52,-13
7,GAS,alpine_f1_team,2,9.0,0,0,8.0,10.5,91.39,0.87,1.68,5
21,VER,red_bull,2,8.0,0,0,11.0,14.0,90.63,1.01,0.91,6
11,LAW,rb_f1_team,2,6.0,0,0,10.0,11.0,91.61,1.10,1.90,2
13,LIN,rb_f1_team,2,4.0,0,0,10.0,12.0,91.73,1.51,2.02,4


In [1]:
# ── Close connection ──────────────────────────────────────────────────────────
con.close()
print("DuckDB connection closed.")

NameError: name 'con' is not defined

---

## Notes

- All queries read from `gold.*` mart tables only — never Silver or Bronze.
- Lap filters applied at load: `is_accurate = TRUE`, `lap_number > 1`,
  `pit_out_time_seconds IS NULL` (proxy for pit outlap — no dedicated flag in this mart).
- `TEAM_COLORS` keyed on `team_id` (lap_times mart) and `constructor` (race_results/standings).
  Both use the same lowercase slug — update if 2026 IDs differ from the palette above.
- Connection opened `read_only=True` — notebook cannot write to DuckDB.
- Re-run after each new round loads — `rounds_loaded` is queried dynamically.

**Next:** `02_team_analysis.ipynb` — constructor pace, pit stop performance, operational excellence.